# Credit Scoring Model Operations

You didn't build this model. You don't know the person who did. They've left the company. You have: the model, some documentation, and three quarters of production data. Your job: figure out if this model is still fit for purpose.

Welcome to most ML engineering jobs.

## 1. Reading the Documentation

Let's start by reading the model card — the only documentation we have.

In [ ]:
with open("./model_card.md") as f:
    print(f.read())

### Questions to consider

- What do we know about the training data? (Not much — "Q4 2022 loan applications" with no size or distribution info)
- What metrics were reported? (Accuracy and AUC on a held-out set — but how was the set composed?)
- What's **not** in this documentation that you'd want to know? (Fairness analysis? Drift plan? Feature distributions? Class balance?)
- Is this documentation adequate for a model making lending decisions?

## 2. Loading and Inspecting the Model

In [ ]:
%pip install seaborn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
from sklearn.calibration import calibration_curve

In [ ]:
pipeline = joblib.load("./model_pipeline.joblib")
print(f"Pipeline steps: {[step[0] for step in pipeline.steps]}")
print(f"Model type: {type(pipeline.named_steps['classifier'])}")

In [ ]:
# Inspect the preprocessing
preprocessor = pipeline.named_steps["preprocessor"]
print("Transformers:")
for name, transformer, columns in preprocessor.transformers_:
    print(f"  {name}: {type(transformer).__name__} \u2192 {columns}")

## 3. Evaluating on Current Data

In [ ]:
q1 = pd.read_csv("./data_q1.csv")
print(f"Q1 shape: {q1.shape}")
print(f"Default rate: {q1['defaulted'].mean():.1%}")

feature_cols = ["income", "employment_length", "loan_amount", "credit_history_length",
                "num_open_accounts", "debt_to_income", "age_group", "region"]
X_q1 = q1[feature_cols]
y_q1 = q1["defaulted"]

In [ ]:
y_pred_q1 = pipeline.predict(X_q1)
y_proba_q1 = pipeline.predict_proba(X_q1)[:, 1]

print(classification_report(y_q1, y_pred_q1))
print(f"AUC: {roc_auc_score(y_q1, y_proba_q1):.3f}")

Compare these to the model card. Has performance changed? (It should be similar since Q1 is close to the training distribution.)

## 4. Drift Detection

Now let's look at how the data — and model performance — changes across quarters.

In [ ]:
q2 = pd.read_csv("./data_q2.csv")
q3 = pd.read_csv("./data_q3.csv")

In [ ]:
# Performance across quarters
for name, data in [("Q1", q1), ("Q2", q2), ("Q3", q3)]:
    X = data[feature_cols]
    y = data["defaulted"]
    y_pred = pipeline.predict(X)
    y_proba = pipeline.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, y_proba)
    f1 = f1_score(y, y_pred)
    print(f"{name}: AUC = {auc:.3f}, F1 = {f1:.3f}, Default rate = {y.mean():.1%}")

In [ ]:
def calculate_psi(expected, actual, bins=10):
    """Calculate Population Stability Index between two distributions."""
    breakpoints = np.linspace(
        min(expected.min(), actual.min()),
        max(expected.max(), actual.max()),
        bins + 1
    )
    expected_counts = np.histogram(expected, breakpoints)[0] / len(expected)
    actual_counts = np.histogram(actual, breakpoints)[0] / len(actual)
    # Avoid division by zero
    expected_counts = np.clip(expected_counts, 0.001, None)
    actual_counts = np.clip(actual_counts, 0.001, None)
    psi = np.sum((actual_counts - expected_counts) * np.log(actual_counts / expected_counts))
    return psi

In [ ]:
# Calculate PSI for numerical features between Q1 and Q3
numerical_features = ["income", "employment_length", "loan_amount",
                      "credit_history_length", "num_open_accounts", "debt_to_income"]

print("PSI (Q1 vs Q3) \u2014 Population Stability Index")
print("< 0.1 = stable | 0.1\u20130.25 = moderate shift | > 0.25 = significant drift\n")
for feat in numerical_features:
    psi = calculate_psi(q1[feat], q3[feat])
    status = "\u2713 stable" if psi < 0.1 else ("\u26a0 moderate" if psi < 0.25 else "\u2717 significant drift")
    print(f"  {feat:30s} PSI = {psi:.3f}  {status}")

In [ ]:
# Visualise the drift for the most shifted features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, ["income", "debt_to_income", "loan_amount"]):
    ax.hist(q1[feat], bins=25, alpha=0.6, density=True, label="Q1")
    ax.hist(q3[feat], bins=25, alpha=0.6, density=True, label="Q3")
    ax.set_title(feat)
    ax.legend()
plt.suptitle("Feature distribution shift: Q1 vs Q3")
plt.tight_layout()
plt.show()

## 5. Fairness Analysis

The model makes lending decisions. UK equality law prohibits discrimination on protected characteristics including age. Does this model treat different age groups equitably?

In [ ]:
# Calculate model performance by age group on Q1 data
# For each age group, calculate: recall, precision, and the approval rate (1 - predicted default rate)

for age_group in sorted(q1["age_group"].unique()):
    mask = q1["age_group"] == age_group
    X_group = q1[mask][feature_cols]
    y_group = q1[mask]["defaulted"]
    pred_group = pipeline.predict(X_group)
    proba_group = pipeline.predict_proba(X_group)[:, 1]

    recall = recall_score(y_group, pred_group, zero_division=0)
    precision = precision_score(y_group, pred_group, zero_division=0)
    approval_rate = (pred_group == 0).mean()
    actual_default = y_group.mean()

    print(f"{age_group:8s} | Recall: {recall:.2f} | Precision: {precision:.2f} | "
          f"Approval rate: {approval_rate:.1%} | Actual default: {actual_default:.1%}")

**Equal opportunity** asks: is the recall (catching actual defaults) similar across groups? If the model is worse at catching defaults for younger applicants, those borrowers are being under-served.

**Demographic parity** asks: is the approval rate similar across groups? If younger applicants are approved at a much lower rate, that might indicate bias — even if the model is technically "correct" based on the features.

### Discussion

1. Are there meaningful disparities in the results above?
2. If `age_group` is a feature the model uses, is that appropriate? Could it be a proxy for discrimination?
3. What about `region` — could that act as a proxy for ethnicity or socioeconomic status?

## 6. Calibration

When the model says "this applicant has a 70% chance of defaulting," is it right 70% of the time? This is calibration. Poorly calibrated models make it impossible to set sensible decision thresholds.

In [ ]:
y_proba_q1 = pipeline.predict_proba(X_q1)[:, 1]
fraction_of_positives, mean_predicted_value = calibration_curve(y_q1, y_proba_q1, n_bins=10)

plt.figure(figsize=(7, 6))
plt.plot(mean_predicted_value, fraction_of_positives, marker="o", label="Model")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Perfectly calibrated")
plt.xlabel("Mean predicted probability")
plt.ylabel("Fraction of positives")
plt.title("Calibration Curve (Q1)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

If the curve lies above the diagonal, the model under-predicts default risk (it says 30% but the real rate is higher). If below, it over-predicts. For lending decisions, miscalibration means the bank is either taking on too much risk or rejecting too many good applicants.

## 7. The Recommendation

You've now assessed this model on four dimensions:

1. **Performance** — how well does it predict defaults?
2. **Drift** — has the data shifted since training?
3. **Fairness** — does it treat demographic groups equitably?
4. **Calibration** — are the predicted probabilities reliable?

Based on your findings, what do you recommend?

- **Option A: Keep as-is with monitoring.** Performance is still acceptable, drift is minimal, fairness is reasonable.
- **Option B: Retrain on recent data.** The model's relationships are still valid but the data has shifted — retraining with Q2/Q3 data would recalibrate it.
- **Option C: Rebuild with different features or model type.** The current model has fundamental issues (e.g. using age directly as a feature, poor fairness properties) that retraining won't fix.
- **Option D: Decommission and replace.** The model is no longer fit for purpose. Remove it from production and design a replacement from scratch.

There is no single right answer, but the reasoning must be sound.

## 8. Decommissioning Considerations

If you recommended replacement (or if the model degrades further in future quarters), decommissioning is not as simple as deleting a file. Consider:

- **Data retention** — what happens to the predictions this model already made? Regulatory requirements may require keeping records for years
- **Model versioning** — the replacement model needs a version number, an updated model card, and a clear record of why the old model was retired
- **Rollback plan** — if the new model performs worse, can you revert? How?
- **Stakeholder communication** — the business, compliance, and customer service teams all need to know what's changing and why
- **Transition period** — running old and new models in parallel (shadow mode) before switching over